<a href="https://colab.research.google.com/github/carms-s-a/nsd_analysis_newspaper/blob/main/analisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este notebook realiza un análisis morfológico, léxico y sintáctico sobre un corpus de transcripciones de noticias. Todos los créditos del material corresponden a John Atkinson-Abutridy.

In [3]:
!git clone https://github.com/carms-s-a/nsd_analysis_newspaper.git
%cd nsd_analysis_newspaper

Cloning into 'nsd_analysis_newspaper'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 107 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 52.80 KiB | 3.77 MiB/s, done.
/content/nsd_analysis_newspaper/nsd_analysis_newspaper


In [4]:
# importar librerías pertinentes
!pip install spacy
!python -m spacy download es_core_news_sm
from nltk.stem import SnowballStemmer
import es_core_news_sm
from spacy.lang.es.stop_words import STOP_WORDS
from spacy import displacy
from pathlib import Path
from string import punctuation
import regex
import re
nlp =es_core_news_sm.load()
stemmer = SnowballStemmer('spanish')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 83.0 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
# Definir función para leer el texto
def leertexto(filename):
	f=open(filename, 'r', encoding='utf-8')
	texto=f.read().split('.')
	f.close()
	return texto

In [6]:
# Definir función de pre-procesamiento
def preprocesar(oraciones):
	texto_limpio =[]
	for texto in oraciones:
		texto=eliminarstopwords(texto)
		texto=eliminarnumeroypuntuacion(texto)
		if len(texto) !=0:
			texto=re.sub(' +', ' ', texto)
			texto_limpio.append(texto)
	return(texto_limpio)

In [7]:
# Definir función para eliminar las stopwords
def eliminarstopwords(oracion):
	tokens=tokenizacion(oracion)
	oracion_filtrada=[]
	for palabra in tokens:
		if palabra not in STOP_WORDS:
			palabra_limpia=palabra.rstrip()
			if len(palabra_limpia) !=0:
				oracion_filtrada.append(palabra_limpia)
	return(lista_a_oracion(oracion_filtrada))

In [8]:
# definir la función para eliminar números y puntuación
def eliminarnumeroypuntuacion(oracion):
	string_numeros = re.sub(r'[\"\"\¿\°\d+]', ' ', oracion)
	return ''.join(c for c in string_numeros if c not in punctuation)

In [9]:
# Definir función para tokenizar
def tokenizar(oracion):
	doc=nlp(oracion)
	tokens= [palabra.text for palabra in doc]
	return(tokens)

In [10]:
# Definir funciones de lematización o stemming(reducción) se elije una
def lematizar(oracion):
	doc=nlp(oracion)
	lemas =[token.lemma_ for token in doc]
	return(lista_a_oracion(lemas))
def stemming(oracion):
	tokens=tokenizacion(oracion)
	stems=[stemmer.stem(palabra) for palabra in tokens]
	return(lista_a_oracion(stems))
def lista_a_oracion(lista):
	return(' '.join(lista))

In [14]:
# Primero corregimos las funciones que tienen llamadas erróneas a 'tokenizacion' en lugar de 'tokenizar'
def tokenizacion(oracion):
    return tokenizar(oracion)

# Modificamos temporalmente leertexto para que soporte latin-1 si falla utf-8
def leertexto_seguro(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            return f.read().split('.')
    except UnicodeDecodeError:
        with open(filename, 'r', encoding='latin-1') as f:
            return f.read().split('.')

# Finalmente se corren todas las funciones
oraciones_originales = leertexto_seguro("data/CORPUS/deportes/d1.txt")
oraciones = preprocesar(oraciones_originales)
texto_reducido = [stemming(oracion) for oracion in oraciones]
texto_lematizado = [lematizar(oracion) for oracion in oraciones]

In [15]:
# Se imprime el texto lematizado y el texto reducido (stemmed)
print(texto_reducido)
print(texto_lematizado)

['el golfist chilen termin jorn tarjet clasific certam disput tex estad unid', 'joaquin niemann logr', 'el golfist chilen pas cort clasific seman charl schwab challeng pga tour', 'el chilen parej tex rendimient rond permit termin tarjet acumul', 'el talagantin deb esper hor avanz depend result rival', 'com cort cerr segu competent', 'niemann registr birdi hoy bogeys bander registr', 'este sab comenz particip hor chil', 'el suec jon blixt termin lid registr birdi hoy aguil bander']
['el golfista chileno terminar jornada tarjeta clasificar certamen disputar Texas Estados Unidos', 'Joaquín Niemann lograr', 'el golfista chileno pasar corte clasificar semana Charles Schwab Challenge PGA Tour', 'el chileno parejo Texas rendimiento ronda permitir terminar tarjeta acumulado', 'el talagantino deber esperar hora avanzar depender resultado rival', 'como corte cerrar seguir competencia', 'Niemann registrar birdi hoyo bogeys bandera registro', 'este sabado comenzar participación hora Chile', 'el su